In [1]:
!pip install spacy

In [3]:
!python -m spacy download en_core_web_md

     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
     -- ------------------------------------- 2.1/33.5 MB 6.5 MB/s eta 0:00:05
     ------ --------------------------------- 5.2/33.5 MB 10.3 MB/s eta 0:00:03
     ---------- ----------------------------- 8.4/33.5 MB 12.7 MB/s eta 0:00:02
     ----------------- --------------------- 15.2/33.5 MB 16.8 MB/s eta 0:00:02
     ------------------------ -------------- 21.2/33.5 MB 18.6 MB/s eta 0:00:01
     ----------------------------- --------- 25.4/33.5 MB 19.9 MB/s eta 0:00:01
     ----------------------------------- --- 30.7/33.5 MB 20.1 MB/s eta 0:00:01
     --------------------------------------- 33.5/33.5 MB 19.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [2]:
import spacy

In [3]:
nlp = spacy.load("en_core_web_md")
# ner_lables = nlp.get_pipe("ner").labels

In [4]:
categories = ["ORG", "PERSON", "LOC", "PRODUCT"]
texts = [
    "Apple is looking at buying U.K. startup for $1 billion",
    "ArithmeticError is a built-in exception in Python",
    "Google is a multinational technology company",
    "Elon Musk is the CEO of SpaceX and Tesla",]

texts2 = [
    "What is the price of 4 bananas?",
    'How much are 16 chairs?',
    "Give me the value of five laptops"
]

In [5]:
docs = [nlp(text) for text in texts]
for doc in docs:
    entities = []
    for ent in doc.ents:
        # if ent.label_ in categories:
        entities.append((ent.text, ent.label_))
    print(f"Entities in '{doc.text}':  {entities}")

Entities in 'Apple is looking at buying U.K. startup for $1 billion':  [('Apple', 'ORG'), ('U.K.', 'GPE'), ('$1 billion', 'MONEY')]
Entities in 'ArithmeticError is a built-in exception in Python':  [('ArithmeticError', 'ORG'), ('Python', 'GPE')]
Entities in 'Google is a multinational technology company':  [('Google', 'ORG')]
Entities in 'Elon Musk is the CEO of SpaceX and Tesla':  [('Elon Musk', 'PERSON'), ('SpaceX', 'ORG'), ('Tesla', 'GPE')]


In [18]:
from spacy.training.example import Example
from spacy.util import minibatch
import random

In [20]:
training_data = [
    ("Apple is looking at buying U.K. startup for $1 billion", 
     {"entities": [(0, 5, "ORG"), (27, 30, "GPE"), (44, 54, "MONEY")]}),

    ("San Francisco considers banning sidewalk delivery robots", 
     {"entities": [(0, 13, "GPE")]}),

    ("London is a big city in the United Kingdom.", 
     {"entities": [(0, 6, "GPE"), (31, 48, "GPE")]}),

    ("Elon Musk founded SpaceX and Tesla.", 
     {"entities": [(0, 9, "PERSON"), (18, 24, "ORG"), (29, 34, "ORG")]}),

    ("Google's headquarters is in Mountain View, California.", 
     {"entities": [(0, 6, "ORG"), (31, 44, "GPE")]}),

    ("Barack Obama was born on August 4, 1961 in Honolulu.", 
     {"entities": [(0, 12, "PERSON"), (26, 40, "DATE"), (44, 52, "GPE")]}),

    ("Amazon acquired Whole Foods for $13.7 billion in 2017.", 
     {"entities": [(0, 6, "ORG"), (16, 27, "ORG"), (33, 47, "MONEY"), (51, 55, "DATE")]}),

    ("Facebook rebranded itself to Meta in 2021.", 
     {"entities": [(0, 8, "ORG"), (28, 32, "ORG"), (36, 40, "DATE")]}),
]

test_data = [
    ("Microsoft released a new product in Redmond.", 
     {"entities": [(0, 9, "ORG"), (37, 44, "GPE")]}),

    ("Sundar Pichai is the CEO of Google.", 
     {"entities": [(0, 13, "PERSON"), (28, 34, "ORG")]}),

    ("Tesla opened a new Gigafactory in Berlin in 2022.", 
     {"entities": [(0, 5, "ORG"), (38, 44, "GPE"), (48, 52, "DATE")]}),

    ("Angela Merkel led Germany for 16 years.", 
     {"entities": [(0, 13, "PERSON"), (18, 25, "GPE")]}),

    ("IBM and Oracle are major tech companies.", 
     {"entities": [(0, 3, "ORG"), (8, 14, "ORG")]}),

    ("Joe Biden visited Kyiv during the summit.", 
     {"entities": [(0, 9, "PERSON"), (18, 22, "GPE")]}),

    ("Meta announced a partnership with Qualcomm.", 
     {"entities": [(0, 4, "ORG"), (34, 42, "ORG")]}),

    ("The Eiffel Tower is a famous landmark in Paris.", 
     {"entities": [(4, 16, "FAC"), (42, 47, "GPE")]}),

    ("Nvidia's shares rose after the keynote in Taipei.", 
     {"entities": [(0, 6, "ORG"), (46, 52, "GPE")]}),

    ("OpenAI unveiled a new version of ChatGPT in 2023.", 
     {"entities": [(0, 6, "ORG"), (34, 41, "PRODUCT"), (45, 49, "DATE")]}),
]



In [ ]:
nlp = spacy.load("en_core_web_md")

if 'ner' not in nlp.pipe_names:
    ner = nlp.add_pipe("ner", last=True)
else:
    ner = nlp.get_pipe("ner")

for _, annotations in training_data:
    for ent in annotations["entities"]:
        if ent[2] not in ner.labels:
            ner.add_label(ent[2])
            
            
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]
with nlp.disable_pipes(*other_pipes):
    optimizer = nlp.begin_training()
    epochs = 30
    for epoch in range(epochs):
        random.shuffle(training_data)
        losses = {}
        batches = minibatch(training_data, size=2)
        for batch in batches:
            examples = []
            for text, annotations in batch:
                doc = nlp.make_doc(text)
                example = Example.from_dict(doc, annotations)
                examples.append(example)
            nlp.update(examples, drop=0.5, losses=losses)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {losses['ner']:.4f}")
        
nlp.to_disk("custom_ner_model")

trained_nlp = spacy.load("custom_ner_model")q
            